> ## SOLUTION / ANSWER KEY &mdash; Lab 5.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-memory-scratchpad.ipynb`](../lab-06-memory-scratchpad.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.6 &mdash; Memory: the Agent's Scratchpad

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Append each (thought, action, observation) step to memory
- Format the memory into the running scratchpad the model re-reads
- Feed the scratchpad to the REAL model and watch it continue from it

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Memory &mdash; short-term &amp; long-term](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-06"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
The agent's **short-term memory** is a **scratchpad**: the running transcript of every Thought, Action and
Observation so far. It is how the agent knows **what it already tried** &mdash; you store steps in a list
and format them back into the text the model re-reads each turn. Below, you build it and then hand it to the
**real** model so it can decide the **next** step instead of restarting. (Long-term memory is a vector
store &mdash; the bridge to RAG.)

In [ ]:
# DEMO -- one step is just a small dict
step = {"thought": "I need the population", "action": "lookup", "observation": "120000"}
print(step)

## Build it
Implement `remember`, `scratchpad`, and `should_stop`.

In [ ]:
def make_memory():
    return []

def remember(memory, thought, action, observation):
    memory.append({"thought": thought, "action": action, "observation": observation})
    return memory

def scratchpad(memory):
    parts = []
    for s in memory:
        t, a, o = s["thought"], s["action"], s["observation"]
        parts.append("Thought: " + str(t) + " | Action: " + str(a) + " | Observation: " + str(o))
    return "\n".join(parts)

def should_stop(memory):
    return any(s["action"] == "final" for s in memory)

try:
    m = make_memory()
    remember(m, "I need the population", "lookup", "120000")
    print(scratchpad(m))
    print("stop?", should_stop(m))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Put one step in memory, then hand the **scratchpad** to the real model and let it propose the next Action &mdash; it can only do that because it read the memory.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        m = make_memory()
        remember(m, "I need the population first", "lookup", "120000")
        prompt = ("Here is the agent's scratchpad so far:\n" + scratchpad(m) +
                  "\n\nThe goal is to report TWICE the population. "
                  "What is the next Action and Action Input? Reply in two short lines.")
        print("MODEL'S NEXT STEP (it READ your scratchpad):\n" + llm_text(prompt))
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The model proposed the **next** step (calculator on 120000) because your **scratchpad** told it the population was already looked up.
- Delete the `remember(...)` line and re-run: with no memory the model has nothing to build on &mdash; that's the whole value of a scratchpad.
- In a framework this is a component you attach; here you built it and fed it to a real model by hand.

## Your turn (open task &mdash; no grader)
Add a second step to memory (the calculator result), re-format the scratchpad, and ask the model for a
**Final Answer**. **What good looks like:** with the fuller scratchpad the model closes out the task; strip
memory and its follow-ups fall apart.

---
### Key takeaway
The scratchpad is the agent's working memory for one task, and a REAL model reads it to continue. Long-term memory (a vector store) arrives with RAG later.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>